# Agents, State & Graph — Built From Scratch

This notebook builds the entire multi-agent system from scratch using raw libraries — no imports from `fin_agents`.

**What we build:**
1. `ResearchState` — the shared TypedDict whiteboard
2. LLM clients — Groq and Gemini via LangChain
3. Orchestrator agent — extracts ticker from query
4. Fundamentals agent — fetches metrics + LLM commentary
5. News agent — fetches headlines + sentiment tagging
6. RAG agent — sub-queries + semantic retrieval
7. Synthesis agent — final markdown brief
8. LangGraph — wires agents into a parallel graph

---
## 0 · Setup

In [1]:
import os, sys, json, hashlib, asyncio
from dotenv import load_dotenv
load_dotenv('../.env')

QUERY = "Analyze Apple's latest quarter"
print('Ready')

Ready


---
## 1 · Shared State

All agents read from and write to a single `TypedDict`. No agent talks directly to another — they all go through this shared whiteboard. LangGraph merges each agent's returned dict back into the state.

In [2]:
from typing import TypedDict

class ResearchState(TypedDict):
    query: str           # original user question
    ticker: str          # extracted by orchestrator
    fundamentals: dict | None   # written by fundamentals agent
    news: list | None           # written by news agent
    news_summary: str | None    # written by news agent
    rag_chunks: list | None     # written by RAG agent
    final_report: str | None    # written by synthesis agent

# Start with only a query — agents fill in the rest
state: ResearchState = {
    'query': QUERY,
    'ticker': '',
    'fundamentals': None,
    'news': None,
    'news_summary': None,
    'rag_chunks': None,
    'final_report': None,
}

print('Initial state keys:', list(state.keys()))
print('Populated fields  :', [k for k, v in state.items() if v])

Initial state keys: ['query', 'ticker', 'fundamentals', 'news', 'news_summary', 'rag_chunks', 'final_report']
Populated fields  : ['query']


---
## 2 · LLM Clients

We use two models:
- **Groq llama-3.3-70b** — fast, free tier, for orchestrator + data agents
- **Groq llama-3.3-70b** — also used for synthesis here (swap to Gemini if you have `GOOGLE_API_KEY`)

In [3]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0)

# Quick sanity check
response = await llm.ainvoke([HumanMessage(content='Reply with just: OK')])
print('LLM check:', response.content)

/Users/minaehab/Desktop/fin-research-agents/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LLM check: OK


---
## 3 · Orchestrator Agent

Reads the raw query, uses the LLM to extract the ticker symbol, writes `ticker` to state.

In [4]:
async def orchestrator(state: ResearchState) -> dict:
    prompt = f"""Extract the stock ticker symbol from this query.
If a company name is given (e.g. Apple, Nvidia), convert it to the correct ticker.
Return ONLY the uppercase ticker — nothing else.

Query: {state['query']}"""

    response = await llm.ainvoke([HumanMessage(content=prompt)])
    ticker = response.content.strip().upper()
    return {'ticker': ticker}


result = await orchestrator(state)
state.update(result)

print(f'Query  : {state["query"]}')
print(f'Ticker : {state["ticker"]}')

Query  : Analyze Apple's latest quarter
Ticker : AAPL


---
## 4 · Fundamentals Agent

Reads `ticker`, fetches metrics from Yahoo Finance, passes them to the LLM for a 1-paragraph commentary.

In [5]:
import yfinance as yf

async def fundamentals_agent(state: ResearchState) -> dict:
    ticker = state['ticker']
    info   = yf.Ticker(ticker).info

    metrics = {
        'name'      : info.get('longName', ticker),
        'sector'    : info.get('sector'),
        'market_cap': info.get('marketCap'),
        'pe_ratio'  : info.get('trailingPE'),
        'eps'       : info.get('trailingEps'),
        'revenue'   : info.get('totalRevenue'),
    }

    prompt = f"""You are a buy-side equity analyst. Write a concise 1-paragraph commentary on these metrics for {metrics['name']} ({ticker}):

Sector    : {metrics['sector']}
Market Cap: ${metrics['market_cap']:,.0f}
P/E Ratio : {metrics['pe_ratio']}
EPS       : {metrics['eps']}
Revenue   : ${metrics['revenue']:,.0f}

Be direct. Highlight valuation signals and any notable observations."""

    response  = await llm.ainvoke([HumanMessage(content=prompt)])
    metrics['commentary'] = response.content.strip()
    return {'fundamentals': metrics}


result = await fundamentals_agent(state)
state.update(result)
f = state['fundamentals']

print(f"{f['name']}  |  Market Cap: ${f['market_cap']:,.0f}  |  P/E: {f['pe_ratio']}")
print(f"\nCommentary:\n{f['commentary']}")

Apple Inc.  |  Market Cap: $3,914,500,145,152  |  P/E: 33.7554

Commentary:
Apple Inc.'s (AAPL) valuation appears stretched, with a P/E ratio of 33.75 indicating a premium to the market. The company's strong earnings per share (EPS) of $7.89 and substantial revenue of $435.6 billion support this valuation to some extent. However, the high P/E multiple suggests that investors have high growth expectations, which may be challenging to sustain. Notably, Apple's market capitalization of nearly $3.9 trillion is a testament to its dominance in the technology sector. Overall, while Apple's financials are impressive, the current valuation may pose a risk to investors if the company fails to deliver on growth expectations, warranting a cautious approach to investing in the stock.


---
## 5 · News Agent

Fetches 5 headlines, makes a single LLM call to tag each with sentiment and produce an overall theme summary.

In [6]:
import requests
from datetime import datetime

async def news_agent(state: ResearchState) -> dict:
    ticker = state['ticker']

    # Fetch headlines
    resp = requests.get(
        'https://newsapi.org/v2/everything',
        params={'q': ticker, 'sortBy': 'publishedAt', 'pageSize': 5, 'language': 'en'},
        headers={'X-Api-Key': os.environ['NEWSAPI_KEY']},
        timeout=15,
    )
    articles = resp.json().get('articles', [])

    if not articles:
        return {'news': [], 'news_summary': 'No recent news found.'}

    headlines = '\n'.join(f'{i+1}. {a["title"]}' for i, a in enumerate(articles))

    # Single LLM call for sentiment + summary
    prompt = f"""For each headline assign a sentiment: positive, negative, or neutral.
Then write a 2-3 sentence summary of the overall news theme.

Headlines:
{headlines}

Respond in this exact JSON format:
{{"sentiments": ["positive", "neutral", ...], "summary": "Overall theme."}}"""

    response = await llm.ainvoke([HumanMessage(content=prompt)])
    try:
        parsed     = json.loads(response.content.strip())
        sentiments = parsed.get('sentiments', [])
        summary    = parsed.get('summary', '')
    except json.JSONDecodeError:
        sentiments = ['neutral'] * len(articles)
        summary    = response.content.strip()

    news_items = []
    for a, sent in zip(articles, sentiments):
        news_items.append({
            'title'    : a['title'],
            'source'   : a['source']['name'],
            'url'      : a['url'],
            'sentiment': sent,
        })

    return {'news': news_items, 'news_summary': summary}


result = await news_agent(state)
state.update(result)

print(f'Theme: {state["news_summary"]}\n')
print('Tagged headlines:')
for a in state['news']:
    print(f'  [{a["sentiment"]:>8}]  {a["title"]}')

Theme: The overall news theme is positive, with multiple headlines indicating a surge in stock market performance, favorable analyst predictions, and new product releases, suggesting a bullish outlook for investors and technology enthusiasts.

Tagged headlines:
  [positive]  2 charts show why Magnificent 7 stocks are being loved again
  [positive]  One Analyst Breaks From The Herd, Says Apple’s DRAM Buying Spree Won’t Crush Margins Despite 2.4 Exabyte Appetite
  [ neutral]  Anker Japan、単ポート最大140W出力でMacBook Proの高速充電も可能なモバイルバッテリー「Anker Prime Power Bank (20100mAh, 220W)」と専用充電ベースのセットモデルを発売。
  [positive]  The S&P 500 has blown past 7,000 in an epic comeback rally. Here’s why it can keep going.
  [positive]  3 ASX ETFs to build a portfolio around in 2026


---
## 6 · RAG Agent

Generates 3 sub-queries from the original question, retrieves the top-5 chunks for each from ChromaDB, deduplicates by content hash.

In [7]:
import chromadb
from chromadb.utils import embedding_functions

# Connect to ChromaDB (use the scratch DB from the data notebook, or the main one)
chroma_client = chromadb.PersistentClient(path='../chroma_db')
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name='BAAI/bge-m3')
col = chroma_client.get_or_create_collection('filings', embedding_function=embed_fn)

async def rag_agent(state: ResearchState) -> dict:
    ticker = state['ticker']
    query  = state['query']

    # Step 1: generate sub-queries
    prompt = f"""Generate exactly 3 specific sub-queries to search SEC filing chunks for:
Original question: {query}
Company: {ticker}
Return ONLY a JSON array of 3 strings."""

    response = await llm.ainvoke([HumanMessage(content=prompt)])
    try:
        sub_queries = json.loads(response.content.strip())
    except Exception:
        sub_queries = [query]

    print(f'Sub-queries generated:')
    for sq in sub_queries:
        print(f'  - {sq}')

    # Step 2: retrieve + deduplicate
    seen, all_chunks = set(), []
    for sq in sub_queries:
        results = col.query(
            query_texts=[sq],
            n_results=5,
            where={'ticker': ticker},
        )
        for doc, meta, dist in zip(
            results['documents'][0],
            results['metadatas'][0],
            results['distances'][0],
        ):
            key = hashlib.md5(doc.encode()).hexdigest()
            if key not in seen:
                seen.add(key)
                all_chunks.append({'text': doc, 'meta': meta, 'score': dist, 'sub_query': sq})

    return {'rag_chunks': all_chunks}


result = await rag_agent(state)
state.update(result)

print(f'\n{len(state["rag_chunks"])} unique chunks retrieved')
for i, c in enumerate(state['rag_chunks'][:3], 1):
    print(f'\n[{i}] score={c["score"]:.4f}  chunk_index={c["meta"]["chunk_index"]}')
    print(c['text'][:200])

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 64026.43it/s]


Sub-queries generated:
  - AAPL quarterly earnings report
  - AAPL latest financial statements
  - AAPL Q4 revenue and net income

7 unique chunks retrieved

[1] score=0.4206  chunk_index=7
2025-09-28 2025-12-27 0000320193 aapl:DeirdreOBrienMember 2025-09-28 2025-12-27 UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 FORM 10-Q (Mark One) ☒ QUARTERLY REPORT PURSUANT

[2] score=0.4521  chunk_index=9
to use the extended transition period for complying with any new or revised financial accounting standards provided pursuant to Section 13(a) of the Exchange Act. ☐ Indicate by check mark whether the 

[3] score=0.4699  chunk_index=17
of net sales. As of December 27, 2025 and September 27, 2025, the Company had total deferred revenue of $ 14.3 billion and $ 13.7 billion, respectively. As of December 27, 2025, the Company expects 66


---
## 7 · Synthesis Agent

Reads the full state — fundamentals, news, and RAG chunks — and produces a structured markdown research brief.

In [8]:
async def synthesis_agent(state: ResearchState) -> dict:
    f      = state['fundamentals'] or {}
    news   = state['news'] or []
    chunks = state['rag_chunks'] or []

    fundamentals_block = f"""Company: {f.get('name')} ({state['ticker']})
Sector    : {f.get('sector')}
Market Cap: ${f.get('market_cap', 0):,.0f}
P/E       : {f.get('pe_ratio')}  |  EPS: {f.get('eps')}
Revenue   : ${f.get('revenue', 0):,.0f}

{f.get('commentary', '')}"""

    news_block = f"Theme: {state.get('news_summary', '')}\n\n"
    news_block += '\n'.join(f"- [{a['sentiment']}] {a['title']}" for a in news)

    rag_block = '\n\n'.join(c['text'][:400] for c in chunks[:8]) or 'No filing excerpts available.'

    prompt = f"""You are a senior equity research analyst. Write a structured markdown research brief.

## Fundamentals
{fundamentals_block}

## Recent News
{news_block}

## SEC Filing Excerpts
{rag_block}

---
Write the brief using EXACTLY these sections:
## Executive Summary
## Key Financials
## Recent News
## Filings Insights
## Risks
## Analyst Takeaway

Be specific, cite numbers, keep each section concise."""

    response = await llm.ainvoke([HumanMessage(content=prompt)])
    return {'final_report': response.content.strip()}


result = await synthesis_agent(state)
state.update(result)
print(f'Report length: {len(state["final_report"])} chars')

Report length: 1920 chars


---
## 8 · LangGraph — Wiring It All Together

So far we've run each agent manually one by one. LangGraph lets us define the flow as a graph and run the three specialist agents **in parallel**, automatically merging their results before synthesis.

```
START → orchestrator → ┬→ fundamentals ─┐
                       ├→ news          ├→ synthesis → END
                       └→ rag          ─┘
                          (parallel)    (barrier)
```

In [9]:
from langgraph.graph import StateGraph, START, END

# Rebuild a fresh initial state
initial = ResearchState(
    query=QUERY,
    ticker='',
    fundamentals=None,
    news=None,
    news_summary=None,
    rag_chunks=None,
    final_report=None,
)

# Build the graph
builder = StateGraph(ResearchState)

builder.add_node('orchestrator',  orchestrator)
builder.add_node('fundamentals',  fundamentals_agent)
builder.add_node('news',          news_agent)
builder.add_node('rag',           rag_agent)
builder.add_node('synthesis',     synthesis_agent)

# Entry
builder.add_edge(START, 'orchestrator')

# Fan-out: orchestrator → 3 parallel agents
builder.add_edge('orchestrator', 'fundamentals')
builder.add_edge('orchestrator', 'news')
builder.add_edge('orchestrator', 'rag')

# Fan-in: all 3 → synthesis (synthesis waits for all three)
builder.add_edge('fundamentals', 'synthesis')
builder.add_edge('news',         'synthesis')
builder.add_edge('rag',          'synthesis')

builder.add_edge('synthesis', END)

graph = builder.compile()
print('Graph compiled')
print('Nodes:', list(graph.nodes.keys()))

Graph compiled
Nodes: ['__start__', 'orchestrator', 'fundamentals', 'news', 'rag', 'synthesis']


In [10]:
import time

start = time.time()
final_state = await graph.ainvoke(initial)
elapsed = time.time() - start

print(f'Completed in {elapsed:.1f}s')
print(f'Ticker      : {final_state["ticker"]}')
print(f'Fundamentals: {final_state["fundamentals"]["name"] if final_state["fundamentals"] else "None"}')
print(f'News items  : {len(final_state["news"] or [])}')
print(f'RAG chunks  : {len(final_state["rag_chunks"] or [])}')
print(f'Report len  : {len(final_state.get("final_report") or "")} chars')

Sub-queries generated:
  - AAPL quarterly earnings report
  - AAPL latest financial statements
  - AAPL Q4 investor presentation
Completed in 3.2s
Ticker      : AAPL
Fundamentals: Apple Inc.
News items  : 5
RAG chunks  : 9
Report len  : 2073 chars


---
## 9 · Final Report

In [11]:
from IPython.display import Markdown, display
display(Markdown(final_state['final_report']))

## Executive Summary
Apple Inc. (AAPL) is trading at a premium valuation with a P/E ratio of 33.76, driven by its sector-leading position and strong brand. The company's financial performance remains robust, with an EPS of $7.89 and revenue of $435.6 billion. Despite the stretched valuation, Apple's consistent track record of innovation and customer loyalty may continue to support its premium multiple.

## Key Financials
- Market Cap: $3,914,571,972,608
- P/E: 33.75602
- EPS: $7.89
- Revenue: $435,617,005,568
These financials indicate a strong and stable company, with a significant market presence.

## Recent News
Recent news themes are predominantly positive, with headlines indicating a surge in stock market performance and favorable analyst predictions. Notable news includes:
- "2 charts show why Magnificent 7 stocks are being loved again"
- "One Analyst Breaks From The Herd, Says Apple’s DRAM Buying Spree Won’t Crush Margins"
These positive trends suggest a bullish outlook for investors and technology companies.

## Filings Insights
According to the latest SEC filing (Form 10-Q), Apple Inc. reported:
- Total deferred revenue of $14.3 billion as of December 27, 2025
- 66% of total deferred revenue expected to be realized in less than a year
- Net income of $42,097 million
These filing insights indicate a strong revenue pipeline and significant profitability.

## Risks
Despite the positive outlook, risks include:
- High valuation multiple, leaving limited room for upside surprise
- Potential margin pressure from increased DRAM buying
- Dependence on consistent innovation and customer loyalty to maintain premium valuation

## Analyst Takeaway
Apple Inc.'s premium valuation is supported by its strong financial performance, sector-leading position, and positive news trends. While risks exist, the company's consistent track record of innovation and customer loyalty may continue to drive growth and support its valuation. As such, we maintain a positive outlook on Apple Inc., but with a cautious eye on potential risks and valuation multiples.